Project Koban: env used will be ml_env_tf_2.15
Goals of this project: 
    Develop a pipeline for image segmentation with minimal assistance from AI agents. 
Structure:
    Object Loading: Object classes (Images, u-net models)
        Considerations: hyperstacks can be very large and memory heavy. I need to use efficent loading methods such as memmap loading to ensure I don't run out of memory. 
        Pathing: flexible pathing that can acomidate local windows or linux system. 
                Points to where the object being loaded lives on the current system. Paths will be defined as variables which will then be called inside of functions. 
                Returns: windows vs linux, file exists. 
                Packages required: Pathlib, os 

        Images: 5D Hyperstacks (dimentions may vary so I should build such that meta data defines the image and the pipeline is flexible to acommodate different dimentions)
            Returns: Image Shape, Dimentions, axes,
                        Expected image shape T,C,Z,X,Y  
                Packages: Tifffile, numpy
        
        U-net Models: 3 class models containing 0 = background, 1 = nucleus, 2 = droplets. .karas format
               Returns: A 512 x 512 probility array 
                Set model path to load. 
            

    Image Segmentation: 
        U-net Classification: 
            Segmentation: u-net model applies to the image.
            Constraints: 

In [1]:
print("Kernal Alive")

Kernal Alive


In [3]:
#### Pakages###
import tifffile as tiff
import numpy as np
import pathlib
import os
from dataclasses import dataclass, field
import platform
import datetime
from pathlib import Path

try:
    import tensorflow as tf
except Exception:
    tf = None 

In [5]:
#### Configuration Section#### 
#------------------------------------------------
# System Configurations Set Mainly for HPC Use
#------------------------------------------------

def get_n_workers() -> int:
    return int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 1))

def config_tensorflow_for_GPU() -> None:
    if tf is None:
        raise RuntimeError(f"Tensorflow is Missing")
    else:
        gpus = tf.config.list_physical_devices("GPU")
        if gpus: 
            for gpu in gpus:
                try:
                    tf.config.experimental.set_memory_growth(gpu, True)
                except RuntimeError:
                    print("gpu already intialized")
                print(gpu)
        else:
            print("No GPU Avalible")
   

#------------------------------------------------
# Smart Root Pathing for HPC and Local Use
#------------------------------------------------
def get_system() -> str: # -> str does not change the function but is a annotation to indicate that the function is expected to return a string
    system = platform.system() # The first action of this function is the create the vatriable system and assign it the vlude from platform.system() which is a function from the platform library that returns a string indicating the current operating system. 
    if system == 'Windows':
        return 'windows'
    elif system == 'Linux':
        return 'linux'
    else:
        raise RuntimeError(f"Unsupported system: {system}")

system = get_system()
print(system)

# System dependant DATA_ROOT
if system == 'windows':
    DATA_ROOT = pathlib.Path("c:/users/cowboy/OneDrive - UAB - The University of Alabama at Birmingham/Projects")
elif system == 'linux':
    DATA_ROOT = pathlib.Path("/home/tdeibert/Projects")
else: 
    raise RuntimeError(f"Unsupported system: {system}")

# Sanity Check to Ensure ROOT exists
if DATA_ROOT.exists():
    print("This Is The Way")
else: 
    raise RuntimeError(f"Path not Found: {DATA_ROOT}")

#------------------------------------------------
# Experimental Configurations Set per Experiment
#------------------------------------------------
@dataclass 
class ExperimentConfig:
    #### Experiment Specific Items #### 
    experiment_name: str = "Control" 
    treatment: str = "none"
    replicate: str = "One"
    channel_map: dict[str,int] = field(default_factory =lambda: {"DIO_Membranes": 0, "mCherry_NLS": 1, "Alexaflour647_NPC": 2})
    pixel_size: float = 0.108  #objective dependant
    z_step_size: float = 1.0 
    project_folder: Path = pathlib.Path("Nuclear_Scaling")
    folder_name: Path = pathlib.Path("Data/Control")
    replicate_folder: Path = pathlib.Path("Extract_One")
    image_name: Path = pathlib.Path("control_extract_1.tif")
    experiment_date: datetime.date = datetime.date(2026, 5, 6) #optional for record keeping
    @property
    def image_path(self) -> Path:
        return Path(DATA_ROOT/self.project_folder/self.folder_name/self.replicate_folder/self.image_name)

#------------------------------------------------
# Pipeline Configurations Set per Pipeline
#------------------------------------------------
@dataclass
class PipelineConfig:
    #### Pathing ####
    model_root: Path = pathlib.Path("/Nuclear_Scaling/Code_Repository/Python/Model_Training/Trained_Models")
    model: Path = pathlib.Path("unet_droplet_nucleus_v6_2_best.keras")
    @property
    def model_path(self) -> Path:
        return Path(DATA_ROOT/self.model_root/self.model)
    #----------------------------------#
    # Halo Analysis #
    #----------------------------------#
    halo_step_px: int = 4
    n_halos: int = 4
    erosion_px: int = 2

    #----------------------------------#
    # Segmentation Variables #
    #----------------------------------#
    #stiched image layout
    tile_rows: int = 2
    tile_cols: int = 3 
    minutes_per_tile: int = 1
    serpentine: bool = True
    
    #Unet Variables
    patch_size: int = 512
    patch_stride: int = 256
    background_class_index: int = 0
    droplet_class_index: int = 1 
    nucleus_class_index: int = 2 
    
    #Probability Threasholds
    prediction_threashold: float = 0.5
    nucleus_probability_threashold: float = 0.3
    
    #Shape Filters 
    min_nucleus_area: int = 50
    max_nucleus_area: int = 600
    nucleus_hole_size: int = 200
    nucleus_opening_radius_px: int = 1
    min_nucleus_circularity: float = 0.8
    batch_size: int = 1 

    #Tracking Variables# 
    z_group_tolerance_um: float = 1.0
    time_track_tolerance_um: float = 5.0
    multi_nucleus_exclusion_um: float = 1.0
    track_dbscan_min_samples: int = 1 
    track_dbsxan_eps_um: float = 6.0
    track_crowded_distance_scale: float = 0.65
    track_area_log_weight: float = 12.0
    track_area_ratio_max: float = 5.0 
    
expcfg = ExperimentConfig()
pipecfg = PipelineConfig()
def output_directory_name(experiment_name:str, date:datetime.date, parent_path:Path) -> str:
        """This function will aquire the vairble experiment_name then append the current date and a counter and return a string in the form expeiment_name_date_count"""
        i = 1
        output_directory_name = f"{experiment_name}_{date}_{i}"
        while Path(parent_path / output_directory_name).exists():
            i = i + 1 
            output_directory_name = f"{experiment_name}_{date}_{i}"
        return output_directory_name

def out_directory_creation(DATA_ROOT:Path, expcfg:ExperimentConfig, pipecfg:PipelineConfig, ) -> Path: 
        """this function will use the path from DATA_ROOT and create a new directory with the name from the output_directory variable and the create three subfolders inside the new directory"""
        parent_path = DATA_ROOT/expcfg.project_folder/expcfg.folder_name/expcfg.replicate_folder
        run_name = output_directory_name(expcfg.experiment_name, expcfg.experiment_date, parent_path)
        output_directory = DATA_ROOT/expcfg.project_folder/expcfg.folder_name/expcfg.replicate_folder/run_name
        if output_directory.exists():
            proceed = input(f"Output Directory Found: {output_directory}\nPress y to Continue with new directory creation or n to cancel new directory creation")
            if proceed.lower() == 'n':
                print("Directory creation cancelled by user.", Path(output_directory))
                return output_directory
        output_directory.mkdir()
        mask_directory = output_directory/"masks"
        mask_directory.mkdir()
        note_directory = output_directory/"notes"
        note_directory.mkdir()
        table_directory = output_directory/"tables"
        table_directory.mkdir()
        print(f"Output Directory Created: {output_directory}")
        return output_directory

output_directory = out_directory_creation(DATA_ROOT, expcfg, pipecfg)

windows
This Is The Way
Output Directory Created: c:\users\cowboy\OneDrive - UAB - The University of Alabama at Birmingham\Projects\Nuclear_Scaling\Data\Control\Extract_One\Control_2026-05-06_3


In [6]:
#### Memmap Image Loading and Saving ####
if expcfg.image_path.exists():
    print(f"Image Found: {expcfg.image_path}")
else:
    raise RuntimeError(f"Image Path Invalid: {expcfg.image_path}")

def load_image_as_memmap(image_path: Path) -> tuple:
    """ Function is designed to load a tiff image as a memmap array. Allows for efficeint large image loading and conserves memory"""
    with tiff.memmap(image_path, mode='r') as memmap_image: 
        axis = memmap_image.axis
        shape = memmap_image.shape
        channels = shape[0] if axis == 'C' else None
        print(f"Image Loaded: {image_path}, Shape: {shape}, Channels: {channels}, Axes: {axis}")
        return memmap_image.copy(), shape, channels, axis
    


Image Found: c:\users\cowboy\OneDrive - UAB - The University of Alabama at Birmingham\Projects\Nuclear_Scaling\Data\Control\Extract_One\control_extract_1.tif


Placeholder markdown

In [ ]:
#### Napari Image Visualization #### 
#fall back to plotlib if napari is not available
try:
    import napari
    from napari.utils import notifications 
except Exception:
    napari = None
    notifications = None

viewer = napari.Viewer() if napari is not None else None
viewer.add_image(raw_image, name="Raw Image") if viewer is not None else print("Napari Not Available, Cannot Display Image")

TypeError: 'WindowsPath' object is not subscriptable